<a href="https://colab.research.google.com/github/1999fello-del/steam-data-analysis-ml/blob/main/notebooks/02_ML_Random_Forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sb
import sklearn


In [ ]:
ruta_archivo = '/content/drive/MyDrive/Mio/games_data_processed (2).csv'
df = pd.read_csv(ruta_archivo)
df.head()

,appid,name,release_date,required_age,price,dlc_count,reviews,metacritic_score,achievements,recommendations,...,tags,pct_pos_total,num_reviews_total,pct_pos_recent,num_reviews_recent,rating_positivo,score_bruto,ratio_exito_pct,tipo_jugador,owners_media
0,730,Counter-Strike 2,2012-08-21,0,0.00,1,NaN,0,1,4401572,...,"{'FPS': 90857, 'Shooter': 65397, 'Multiplayer'...",86,8632939,82,96473,0.868255,6.022352,NaN,Multijugador,150000000.0
1,578080,PUBG: BATTLEGROUNDS,2017-12-21,0,0.00,0,NaN,0,37,1732007,...,"{'Survival': 14838, 'Shooter': 12727, 'Battle ...",59,2513842,68,16720,0.592247,3.790584,NaN,Multijugador,75000000.0
2,570,Dota 2,2013-07-09,0,0.00,2,“A modern multiplayer masterpiece.” 9.5/10 – D...,90,0,14337,...,"{'Free to Play': 59933, 'MOBA': 20158, 'Multip...",81,2452595,80,29366,0.815765,5.212436,NaN,Multijugador,350000000.0
3,271590,Grand Theft Auto V Legacy,2015-04-13,17,0.00,0,NaN,96,77,1803063,...,"{'Open World': 32644, 'Action': 23539, 'Multip...",87,1803832,92,17517,0.873088,5.462209,NaN,Multijugador,75000000.0
4,359550,Tom Clancy's Rainbow Six® Siege,2015-12-01,17,3.99,9,NaN,0,0,1165929,...,"{'FPS': 9831, 'PvP': 9162, 'e-sports': 9072, '...",84,1168020,76,12608,0.840691,5.100851,NaN,Multijugador,35000000.0


In [ ]:
print(df.columns)

Index(['appid', 'name', 'release_date', 'required_age', 'price', 'dlc_count',
       'reviews', 'metacritic_score', 'achievements', 'recommendations',
       'supported_languages', 'full_audio_languages', 'packages', 'developers',
       'publishers', 'categories', 'genres', 'user_score', 'positive',
       'negative', 'estimated_owners', 'average_playtime_forever',
       'average_playtime_2weeks', 'median_playtime_forever',
       'median_playtime_2weeks', 'peak_ccu', 'tags', 'pct_pos_total',
       'num_reviews_total', 'pct_pos_recent', 'num_reviews_recent',
       'rating_positivo', 'score_bruto', 'ratio_exito_pct', 'tipo_jugador',
       'owners_media'],
      dtype='object')


Agrupamos los juegos por meses, para luego hacer el M.L.

In [ ]:

# 1. Convertir la columna a formato fecha, limpieza de datos simple para que no haya errores después.
df['release_date'] = pd.to_datetime(df['release_date'])

# 2. "Amontonar" por mes y contar cuántos juegos hay, lo más sencillo para luego poder hacer una visualización en Power BI.
df_mensual = df.resample('MS', on='release_date').size().reset_index(name='cantidad_juegos')

# 3. CREAR LAS NUEVAS COLUMNAS
# mes_ordinal: Un número secuencial (0, 1, 2, 3...) para que el modelo entienda el paso del tiempo.
# El modelo lo puedo hacer sin esta columna, pero puede llegar a no entender la repetición de 12 por cada año, por ello al ser 1,2,3,4,... lo entiende mejor. COMO UN INDICE.
df_mensual['mes_ordinal'] = range(len(df_mensual))

# mes_del_año: El número del mes (1 para enero, 12 para diciembre) para captar la estacionalidad, también
# COMO SIEMPRE LO MEJOR ES PONER LAS COSAS CLARAS Y ORDENADAS AL PRINCIPIO, PARA QUE LUEGO NO HAYA NINGUN ERROR EN EL PROCESO.
df_mensual['mes_del_año'] = df_mensual['release_date'].dt.month

# EXTRA: año: También ayuda mucho al modelo saber en qué año estamos.
df_mensual['año'] = df_mensual['release_date'].dt.year

# Ver el resultado
print(df_mensual.head())

  release_date  cantidad_juegos  mes_ordinal  mes_del_año   año
0   1997-06-01                1            0            6  1997
1   1997-07-01                0            1            7  1997
2   1997-08-01                0            2            8  1997
3   1997-09-01                0            3            9  1997
4   1997-10-01                0            4           10  1997


In [ ]:
df_mensual

,release_date,cantidad_juegos,mes_ordinal,mes_del_año,año
0,1997-06-01,1,0,6,1997
1,1997-07-01,0,1,7,1997
2,1997-08-01,0,2,8,1997
3,1997-09-01,0,3,9,1997
4,1997-10-01,0,4,10,1997
...,...,...,...,...,...
329,2024-11-01,1740,329,11,2024
330,2024-12-01,1587,330,12,2024
331,2025-01-01,1469,331,1,2025
332,2025-02-01,1409,332,2,2025


# EMPEZAMOS CON EL MACHINE LEARNING SIMPLE

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Definimos nuestras características (X) y nuestro objetivo (y)
# Usamos las columnas que creamos antes
X = df_mensual[['mes_ordinal', 'mes_del_año', 'año']] # Lo que el modelo usa para aprender. Se lo conoce como Caracteristicas.
y = df_mensual['cantidad_juegos'] # Lo que el modelo va a predecir. Y a esto como Objetivos.

# Creamos el "cerebro" del modelo
# n_estimators=100 significa que usará 100 "árboles" para decidir su respuesta. Más 'arboles' menos errores en la estimación, PERO ESTE MODELO TEINE UN GRAN PROBLEMA PARA NUESTRO CASO. 'MUY ENCORSETADO'
modelo = RandomForestRegressor(n_estimators=100, random_state=42)

# Entrenamos: aquí es donde el modelo estudia tus 10 años de datos
modelo.fit(X, y)

print("¡Modelo entrenado con éxito!")

¡Modelo entrenado con éxito!


In [ ]:
# Buscamos dónde se quedó nuestro contador de meses (mes_ordinal)
ultimo_mes_ordinal = df_mensual['mes_ordinal'].max()

# Creamos los datos para los 9 meses que faltan (Abril=4, Mayo=5... Diciembre=12)
datos_futuros = {
    'mes_ordinal': range(ultimo_mes_ordinal + 1, ultimo_mes_ordinal + 10),
    'mes_del_año': [4, 5, 6, 7, 8, 9, 10, 11, 12],
    'año': [2025] * 9
}

df_futuro = pd.DataFrame(datos_futuros)

In [ ]:
# Le pedimos al modelo que prediga
predicciones = modelo.predict(df_futuro)

# Agregamos las predicciones a nuestra tabla del futuro para verlas bien
df_futuro['prediccion_juegos'] = predicciones.round(0).astype(int)

# Para que sea más fácil de leer, le ponemos nombre a los meses
nombres_meses = ['Abril', 'Mayo', 'Junio', 'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']
df_futuro['nombre_mes'] = nombres_meses

print("--- ESTIMACIONES PARA EL RESTO DE 2025 ---")
print(df_futuro[['nombre_mes', 'prediccion_juegos']])

--- ESTIMACIONES PARA EL RESTO DE 2025 ---
   nombre_mes  prediccion_juegos
0       Abril                899
1        Mayo                909
2       Junio                903
3       Julio                913
4      Agosto                944
5  Septiembre                952
6     Octubre               1012
7   Noviembre               1023
8   Diciembre               1001


Ahora tenemos dos dataframes uno con los meses y la cantidad de los juegos que sales y otro con la preduccion que nos a propuesto el modelo entrenado.
Ahora hay que unificar las dos columnas para concatenarlos y ver todos los datos.

In [ ]:
# 1. Renombramos la columna de predicción para que coincida con la original
df_futuro_rename = df_futuro.rename(columns={'prediccion_juegos': 'cantidad_juegos'})

# 2. Unimos los dos DataFrames
# Nota: pd.concat "apila" el df_futuro debajo del df_mensual
df_total = pd.concat([df_mensual, df_futuro_rename], ignore_index=True)

# 3. (Opcional) Creamos una columna para distinguir qué es real y qué es predicción
# Esto nos servirá para que la gráfica tenga colores distintos
df_mensual['tipo'] = 'Real'
df_futuro_rename['tipo'] = 'Predicción'
df_total = pd.concat([df_mensual, df_futuro_rename], ignore_index=True)

In [ ]:
df_total

,release_date,cantidad_juegos,mes_ordinal,mes_del_año,año,tipo,nombre_mes
0,1997-06-01,1,0,6,1997,Real,NaN
1,1997-07-01,0,1,7,1997,Real,NaN
2,1997-08-01,0,2,8,1997,Real,NaN
3,1997-09-01,0,3,9,1997,Real,NaN
4,1997-10-01,0,4,10,1997,Real,NaN
...,...,...,...,...,...,...,...
338,NaT,944,338,8,2025,Predicción,Agosto
339,NaT,952,339,9,2025,Predicción,Septiembre
340,NaT,1012,340,10,2025,Predicción,Octubre
341,NaT,1023,341,11,2025,Predicción,Noviembre


In [ ]:
from google.colab import files

# 1. Guardamos el DataFrame en un archivo CSV dentro de Colab
# Usamos index=False para que no cree una columna extra con los números de fila
df_total.to_csv('predicciones_juegos_2025.csv', index=False)

# 2. Descargamos el archivo a nuestro ordenador
# files.download('predicciones_juegos_2025.csv')